## RQ4: What are the most occurring ingredients in NOVA 3 and NOVA 4 that lead to health problems?

The correlation between health risks and higher NOVA 3 (processed) and 
NOVA 4 (ultra-processed) consumption can develop potential health risk(s) 
as shown in Q3. This research question further explores the root cause of 
these illnesses among Americans where ingredients tell the truth - utilizing 
visualizations to show the occurring NOVA 3 and NOVA 4 ingredients found 
in different food categories that lead to health problems.

In [1]:
import pandas as pd
from collections import Counter

df = pd.read_csv("branded_food_cleaned_classified.csv")
print(f"Loaded: {len(df):,} rows")

Loaded: 1,963,676 rows


In [2]:
df_nova3 = df[df["nova_category"] == "NOVA 3 - Processed"].copy()
df_nova4 = df[df["nova_category"] == "NOVA 4 - Ultra-Processed"].copy()

print(f"NOVA 3 products: {len(df_nova3):,}")
print(f"NOVA 4 products: {len(df_nova4):,}")

NOVA 3 products: 325,482
NOVA 4 products: 1,076,861


In [3]:
df_upf = df[df["nova_category"].isin(
    ["NOVA 3 - Processed", "NOVA 4 - Ultra-Processed"]
)].copy()
print(f"Combined NOVA 3 + 4 products: {len(df_upf):,}")

def clean_ingredient(ing):
    ing = ing.strip().upper()
    ing = ing.rstrip(")].,:;")
    ing = ing.lstrip("(")
    return ing.strip()

def split_ingredients(text):
    text = str(text)
    colon = text.find(":")
    comma = text.find(",")
    if colon != -1 and (comma == -1 or colon < comma):
        text = text[colon + 1:]
    parts = {clean_ingredient(p) for p in text.split(",")}
    return [p for p in parts if p and len(p) > 1]
    
work = df_upf[["ingredients", "branded_food_category", "nova_category"]].dropna(subset=["ingredients"]).copy()
work["ingredient"] = work["ingredients"].apply(split_ingredients)
exploded = work.explode("ingredient").dropna(subset=["ingredient"])

# total product count per ingredient
counts = (exploded["ingredient"].value_counts()
          .rename_axis("ingredient").reset_index(name="product_count"))
counts["pct_of_products"] = (counts["product_count"] / len(df_upf) * 100).round(2)

# split by NOVA tier
nova_split = (exploded.groupby(["ingredient", "nova_category"]).size()
              .unstack(fill_value=0)
              .rename(columns={"NOVA 3 - Processed": "nova_3_count",
                               "NOVA 4 - Ultra-Processed": "nova_4_count"})
              .reset_index())

# dominant category — vectorized, fast version (no lambda)
pair_counts = (exploded.groupby(["ingredient", "branded_food_category"])
               .size().reset_index(name="n"))
dominant_cat = (pair_counts.sort_values(["ingredient", "n"], ascending=[True, False])
                .drop_duplicates(subset="ingredient", keep="first")
                [["ingredient", "branded_food_category"]]
                .rename(columns={"branded_food_category": "top_category"}))

# combine into one full summary table
ingredient_summary = (counts
                      .merge(nova_split, on="ingredient", how="left")
                      .merge(dominant_cat, on="ingredient", how="left"))

# export
ingredient_summary.to_csv("nova34_ingredient_summary.csv", index=False)
print(f"Exported {len(ingredient_summary):,} ingredients to nova34_ingredient_summary.csv")
ingredient_summary.head(20)

Combined NOVA 3 + 4 products: 1,402,343
Exported 288,532 ingredients to nova34_ingredient_summary.csv


,ingredient,product_count,pct_of_products,nova_3_count,nova_4_count,top_category
0,SALT,793244,56.57,175601,617643,Cheese
1,SUGAR,625838,44.63,82666,543172,Candy
2,WATER,501787,35.78,149210,352577,Breads & Buns
3,CITRIC ACID,321660,22.94,46164,275496,Candy
4,FOLIC ACID,249138,17.77,41345,207793,Cookies & Biscuits
5,NATURAL FLAVOR,233572,16.66,36557,197015,Ice Cream & Frozen Yogurt
6,NIACIN,223079,15.91,38457,184622,Cookies & Biscuits
7,CORN SYRUP,216523,15.44,8459,208064,Candy
8,SOY LECITHIN,210566,15.02,23252,187314,"Cakes, Cupcakes, Snack Cakes"
9,RIBOFLAVIN,198462,14.15,31205,167257,"Cakes, Cupcakes, Snack Cakes"


In [4]:
df.to_csv('top_ingredients.csv', index=False)

# Limitations
This study has several limitations that shape how its findings should be read. First, the dataset reflects the products available on store shelves rather than what Americans actually eat, and each product is counted only once regardless of how much it sells, so ingredient prevalence does not necessarily reflect real dietary exposure. Second, ingredient lists are qualitative: they reveal that a substance is present but not in what amount, and many ingredients appear only in trace quantities, so frequency alone cannot establish that an ingredient causes harm. Any health implications must therefore be grounded in peer-reviewed epidemiological research, not in ingredient counts. Third, the NOVA classification was applied using keyword-based heuristics rather than validated by trained coders, meaning some products may be miscategorized, particularly at the boundary between NOVA 3 and NOVA 4. Finally, the data covers only branded, packaged foods submitted to GDSN and excludes fresh produce, restaurant meals, and other unpackaged foods, so it represents an important but incomplete slice of the overall American diet.